In [1]:
import librosa
import matplotlib.pyplot as plt
import soundfile as sf

import torch
from IPython.display import Audio
import random
from safetensors.torch import load_file
from models.demucs_equalizer import DemucsEqualizer, DoubleDemucsEqualizer

In [2]:
import numpy as np
import librosa
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr
# from pesq import pesq

def mse_time_domain(reference, estimated):
    # import numpy as np
    reference = (reference - reference.min()) / (reference.max() - reference.min())
    estimated = (estimated - estimated.min()) / (estimated.max() - estimated.min())
    return np.mean((reference - estimated) ** 2)

def mse_freq_domain(reference, estimated):
    # import numpy as np
    delta = 1e-7 
    reference_db = librosa.amplitude_to_db(np.abs(librosa.stft(reference)))
    estimated_db = librosa.amplitude_to_db(np.abs(librosa.stft(estimated)))
    reference_db = (reference_db - reference_db.min()) / (reference_db.max() - reference_db.min())
    estimated_db = (estimated_db - estimated_db.min()) / (estimated_db.max() - estimated_db.min())
    return np.mean((reference_db - estimated_db) ** 2)

def sdr(references, estimates):
    # import numpy as np
    delta = 1e-7 
    num = np.sum(np.square(references))
    den = np.sum(np.square(references - estimates))
    num += delta
    den += delta
    return 10 * np.log10(num / den)

def sisnr(reference, estimated):
    # import numpy as np
    reference = reference - np.mean(reference)
    estimated = estimated - np.mean(estimated)
    reference_energy = np.sum(reference ** 2)
    projection = np.sum(reference * estimated) * reference / reference_energy
    noise = estimated - projection
    si_snr = 10 * np.log10(np.sum(projection ** 2) / np.sum(noise ** 2))
    return si_snr

def new_pesq(reference, estimated, sr=16000):
    pesq(sr,reference, estimated,'wb')

# def cos_sim(reference, estimated):
#     # from sklearn.metrics.pairwise import cosine_similarity
#     similarity = cosine_similarity(reference.reshape(1, -1),estimated.reshape(1, -1))
#     return similarity[0][0]

# def pearson(reference, estimated):
#     # from scipy.stats import pearsonr
#     correlation, _ = pearsonr(reference, estimated)
#     return correlation

def calculate_average(reference_list, estimated_list,function):
    metrix_values = [function(reference, estimated) for reference, estimated in zip(reference_list, estimated_list)]
    average_value = np.mean(metrix_values)
    return average_value

In [3]:
# you can normalize the time domain or frequency domain before calculate the metrics

def normalization(x,target=40):
    # normalize the frequence domain into range (-40,40), but the time domain may exceed (-1,1)
    x_amplitude = np.abs(librosa.stft(x))
    x_spec = librosa.amplitude_to_db(x_amplitude)
    scaling_factor = 10 ** ((target - np.max(x_spec)) / 20)
    return x*scaling_factor

def normalization_to_1(x):
    # # normalize the max of time domain to -1 or 1
    return x*(1/np.max(np.abs(x)))

In [4]:
def split_audio(audio,sr=44100,total_len=200,segment_len=5):
    total_samples = total_len*sr
    segment_samples = segment_len*sr
    segments_list = [audio[i:i + segment_samples] for i in range(0, total_samples, segment_samples)]
    return segments_list

def join_audio(segments: list):
    import numpy as np
    return np.concatenate(segments, axis=0)

In [7]:
def get_prediction_from_audio(seq,x_path = 'data/v1/EN_x/',y1_path = 'data/v1/EN_y1/',y2_path = 'data/v1/EN_y2_headphone/'):
    device = 'cpu'
    x,sr = librosa.load(x_path+f'{seq}.wav',sr=None)
    y1,sr = librosa.load(y1_path+f'{seq}.wav',sr=None)
    y2,sr = librosa.load(y2_path+f'{seq}.wav',sr=None)
    x_list = split_audio(x)
    y1_list = split_audio(y1)
    y2_list = split_audio(y2)
    y1_predict_list,gx_list,y2_predict_list = [],[],[]
    
    checkpoint_path = "assets/V2-full/stage1/pytorch_model.bin"
    model = DemucsEqualizer(device=device)
    state_dict = torch.load(checkpoint_path, map_location=device)
    state_dict = {k[6:]: v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    model.eval()  
    model = model.to(device)
    for i in range(len(x_list)):
        with torch.no_grad():
            y1_predict = model(torch.tensor(x_list[i]).unsqueeze(0).unsqueeze(0)).squeeze(0).squeeze(0).cpu().numpy()
            y1_predict_list.append(y1_predict)

    checkpoint_path = "assets/checkpoint-10419/pytorch_model.bin"
    model = DoubleDemucsEqualizer("assets/V2-full/stage1/pytorch_model.bin", device=device)
    state_dict = torch.load(checkpoint_path, map_location=device)
    state_dict = {k[6:]: v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    model.eval()
    model = model.to(device)
    for i in range(len(x_list)):
        input_waveform = torch.tensor(x_list[i]).unsqueeze(0).unsqueeze(0)
        if len(torch.tensor(x_list[seq]).shape) == 2:
            input_waveform = input_waveform.unsqueeze(1)
            input_waveform = torch.cat([input_waveform, input_waveform], dim=1)       
        input_waveform = torch.cat([input_waveform, input_waveform], dim=1)        
        with torch.no_grad():
            first_waveform = model.model2(input_waveform)
            first_waveform = first_waveform.mean(dim=1)[:,0,:]  
            first_waveform = first_waveform.reshape(first_waveform.shape[0], -1)
            gx_list.append(first_waveform.squeeze(0).cpu().numpy())
            second_waveform = model.model1(first_waveform).squeeze(0).cpu().numpy()  
            y2_predict_list.append(second_waveform)
    return x_list,y1_list,y1_predict_list,y2_list,gx_list,y2_predict_list


In [ ]:
x_list,y1_list,y1_predict_list,y2_list,gx_list,y2_predict_list=get_prediction_from_audio(1)

In [9]:
gx = join_audio(gx_list)
sf.write('./gx-v2-full.wav',gx,44100)

In [ ]:

reference_list = ['x_list','y1_list','y2_list']
estimated_list = ['x_list','y1_list','y1_predict_list','gx_list','y2_list','y2_predict_list']
for function in mse_time_domain,sdr,sisnr:
    print(f'============================================={function.__name__}=============================================')
    for i, (reference) in enumerate([x_list,y1_list,y2_list]):
        for j, (estimated) in enumerate([x_list,y1_list,y1_predict_list,gx_list,y2_list,y2_predict_list]):
            print(reference_list[i],estimated_list[j],calculate_average(reference,estimated,function))
    print('=========================================================================================================\n\n')